In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

print("Initializing capacitor problem...")

# Grid and iteration settings
Nmax = 200
max_iter = 20
tolerance = 1e-3

# Initialize potential grid
V = np.zeros((Nmax, Nmax), dtype=float)

# Boundary conditions for a parallel-plate style capacitor
V[:, 0] = 100.0   # Left plate at 100 V
V[:, -1] = 0.0    # Right plate at 0 V

print("Solving Laplace's equation using Jacobi relaxation...")

# Jacobi relaxation
for iter_count in range(max_iter):
    V_old = V.copy()

    for i in range(1, Nmax - 1):
        for j in range(1, Nmax - 1):
            V[i, j] = 0.25 * (
                V_old[i + 1, j] +
                V_old[i - 1, j] +
                V_old[i, j + 1] +
                V_old[i, j - 1]
            )

    # Reapply boundary conditions
    V[:, 0] = 100.0
    V[:, -1] = 0.0

    # Convergence check
    error = np.max(np.abs(V - V_old))
    if iter_count % 100 == 0:
        print(f"Iteration {iter_count}, max change = {error:.6e}")

    if error < tolerance:
        print(f"Converged after {iter_count} iterations.")
        break

# Create coordinate grid
x = np.arange(0, Nmax)
y = np.arange(0, Nmax)
X, Y = np.meshgrid(x, y)

# 3D surface plot of the potential
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, V, cmap='viridis', edgecolor='none')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('Potential V(x,y)')
ax.set_title('Electrostatic Potential from Laplace Equation')

plt.tight_layout()
plt.show()

# Electric field from the potential
Ey, Ex = np.gradient(-V)

plt.figure(figsize=(8, 6))
plt.contourf(X, Y, V, levels=30, cmap='viridis')
plt.colorbar(label='Potential (V)')
plt.quiver(X[::5, ::5], Y[::5, ::5], Ex[::5, ::5], Ey[::5, ::5], color='white')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Equipotential Contours and Electric Field')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Grid parameters
N = 100  # Grid size
h = 1.0 # Grid spacing
V = np.zeros((N, N, N))  # Potential grid
rho = np.zeros((N, N, N)) # Charge density grid
Ex = np.zeros((N, N, N)) # E-field x-component
Ey = np.zeros((N, N, N)) # E-field y-component
Ez = np.zeros((N, N, N)) # E-field z-component

# Torus parameters
R = 30  # Major radius
r = 8   # Minor radius
sigma_over_epsilon0 = 10.0 # Surface charge density / permittivity

# Create a boolean mask for the torus surface
x = np.arange(N) - N/2
y = np.arange(N) - N/2
z = np.arange(N) - N/2
xx, yy, zz = np.meshgrid(x, y, z)

# Approximate surface of torus (within one grid cell)
dist_sq = (np.sqrt(xx**2 + yy**2) - R)**2 + zz**2
torus_surface_mask = (dist_sq >= (r-h/2)**2) & (dist_sq <= (r+h/2)**2)
rho[torus_surface_mask] = sigma_over_epsilon0 / h # Approximate volume density

# Jacobi relaxation for Poisson's Equation
max_iter = 1000 # Increased iterations for convergence
for _ in range(max_iter):
    V_new = V.copy()
    V_new[1:-1, 1:-1, 1:-1] = (V[:-2, 1:-1, 1:-1] + V[2:, 1:-1, 1:-1] +
                               V[1:-1, :-2, 1:-1] + V[1:-1, 2:, 1:-1] +
                               V[1:-1, 1:-1, :-2] + V[1:-1, 1:-1, 2:] +
                               (h**2) * rho[1:-1, 1:-1, 1:-1]) / 6.0
    V = V_new

# Calculate Electric Field (E = -gradient(V))
Ex[1:-1, 1:-1, 1:-1] = -(V[2:, 1:-1, 1:-1] - V[:-2, 1:-1, 1:-1]) / (2*h)
Ey[1:-1, 1:-1, 1:-1] = -(V[1:-1, 2:, 1:-1] - V[1:-1, :-2, 1:-1]) / (2*h)
Ez[1:-1, 1:-1, 1:-1] = -(V[1:-1, 1:-1, 2:] - V[1:-1, 1:-1, :-2]) / (2*h)

# Plotting
slice_idx = N // 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Potential plot
im = ax1.imshow(V[:, :, slice_idx], cmap='viridis', origin='lower',
                extent=[-N/2, N/2, -N/2, N/2])
fig.colorbar(im, ax=ax1, label='Potential (V)')
ax1.set_title(f'Potential in the XY plane at z={z[slice_idx]}')
ax1.set_xlabel('x')
ax1.set_ylabel('y')

# Electric field plot
skip = 3 # Plot every `skip` vector
ax2.quiver(xx[::skip, ::skip, slice_idx], yy[::skip, ::skip, slice_idx],
           Ex[::skip, ::skip, slice_idx], Ey[::skip, ::skip, slice_idx],
           np.sqrt(Ex[::skip, ::skip, slice_idx]**2 + Ey[::skip, ::skip, slice_idx]**2),
           cmap='Reds')
ax2.set_title(f'Electric Field in the XY plane at z={z[slice_idx]}')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# 3D Visualization
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot a subset of vectors to avoid clutter
skip = 8
quiver_scale = 4 # Adjust this to scale the arrows

ax.quiver(xx[::skip, ::skip, ::skip], yy[::skip, ::skip, ::skip], zz[::skip, ::skip, ::skip],
          Ex[::skip, ::skip, ::skip], Ey[::skip, ::skip, ::skip], Ez[::skip, ::skip, ::skip],
          length=quiver_scale, normalize=True, color='r', alpha=0.6)

# Create a surface for the torus for visualization
u = np.linspace(0, 2 * np.pi, 30)
v = np.linspace(0, 2 * np.pi, 30)
u, v = np.meshgrid(u, v)
x_torus = (R + r * np.cos(v)) * np.cos(u)
y_torus = (R + r * np.cos(v)) * np.sin(u)
z_torus = r * np.sin(v)

ax.plot_surface(x_torus, y_torus, z_torus, color='b', alpha=0.3)


ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Electric Field of a Torus')

# Set aspect ratio to be equal and zoom out
zoom_factor = 1.5
max_range = np.array([xx.max()-xx.min(), yy.max()-yy.min(), zz.max()-zz.min()]).max() / 2.0 * zoom_factor
mid_x = (xx.max()+xx.min()) * 0.5
mid_y = (yy.max()+yy.min()) * 0.5
mid_z = (zz.max()+zz.min()) * 0.5
ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)

plt.show()

In [ ]:
from scipy.interpolate import RegularGridInterpolator
from scipy.integrate import solve_ivp

# Add a sphere with opposite charge density
sphere_r = 3 # Radius of the sphere
sphere_mask = xx**2 + yy**2 + zz**2 <= sphere_r**2
rho[sphere_mask] = -40*sigma_over_epsilon0 / h # Opposite charge density

# --- Recalculate Potential and E-Field with the sphere ---

# Jacobi relaxation for Poisson's Equation
V_sphere = np.zeros((N, N, N)) # New potential grid
for _ in range(max_iter): # Using max_iter from the previous cell
    V_new = V_sphere.copy()
    V_new[1:-1, 1:-1, 1:-1] = (V_sphere[:-2, 1:-1, 1:-1] + V_sphere[2:, 1:-1, 1:-1] +
                               V_sphere[1:-1, :-2, 1:-1] + V_sphere[1:-1, 2:, 1:-1] +
                               V_sphere[1:-1, 1:-1, :-2] + V_sphere[1:-1, 1:-1, 2:] +
                               (h**2) * rho[1:-1, 1:-1, 1:-1]) / 6.0
    V_sphere = V_new

# Calculate Electric Field (E = -gradient(V))
Ex_sphere = np.zeros((N, N, N))
Ey_sphere = np.zeros((N, N, N))
Ez_sphere = np.zeros((N, N, N))
Ex_sphere[1:-1, 1:-1, 1:-1] = -(V_sphere[2:, 1:-1, 1:-1] - V_sphere[:-2, 1:-1, 1:-1]) / (2*h)
Ey_sphere[1:-1, 1:-1, 1:-1] = -(V_sphere[1:-1, 2:, 1:-1] - V_sphere[1:-1, :-2, 1:-1]) / (2*h)
Ez_sphere[1:-1, 1:-1, 1:-1] = -(V_sphere[1:-1, 1:-1, 2:] - V_sphere[1:-1, 1:-1, :-2]) / (2*h)


# --- Plotting ---

# 2D plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

im = ax1.imshow(V_sphere[:, :, slice_idx], cmap='viridis_r', origin='lower',
                extent=[-N/2, N/2, -N/2, N/2])
fig.colorbar(im, ax=ax1, label='Potential (V)')
ax1.set_title(f'Potential in XY plane with Sphere')
ax1.set_xlabel('x')
ax1.set_ylabel('y')

skip = 3
ax2.streamplot(x[::skip], y[::skip],
               Ex_sphere[::skip, ::skip, slice_idx].T, Ey_sphere[::skip, ::skip, slice_idx].T,
               color=np.sqrt(Ex_sphere[::skip, ::skip, slice_idx].T**2 + Ey_sphere[::skip, ::skip, slice_idx].T**2),
               cmap='Reds')
ax2.set_title(f'E-Field in XY plane with Sphere')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_aspect('equal')
plt.tight_layout()
plt.show()

# 3D Visualization
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Torus surface
u = np.linspace(0, 2 * np.pi, 30)
v = np.linspace(0, 2 * np.pi, 30)
u, v = np.meshgrid(u, v)
x_torus = (R + r * np.cos(v)) * np.cos(u)
y_torus = (R + r * np.cos(v)) * np.sin(u)
z_torus = r * np.sin(v)
ax.plot_surface(x_torus, y_torus, z_torus, color='b', alpha=0.6)

# Sphere surface
phi = np.linspace(0, 2 * np.pi, 30)
theta = np.linspace(0, np.pi, 30)
phi, theta = np.meshgrid(phi, theta)
x_sphere = sphere_r * np.sin(theta) * np.cos(phi)
y_sphere = sphere_r * np.sin(theta) * np.sin(phi)
z_sphere = sphere_r * np.cos(theta)
ax.plot_surface(x_sphere, y_sphere, z_sphere, color='b', alpha=0.5)

# Streamlines using RK45
x_grid = np.arange(N) - N / 2
y_grid = np.arange(N) - N / 2
z_grid = np.arange(N) - N / 2

interp_Ex = RegularGridInterpolator((x_grid, y_grid, z_grid), Ex_sphere)
interp_Ey = RegularGridInterpolator((x_grid, y_grid, z_grid), Ey_sphere)
interp_Ez = RegularGridInterpolator((x_grid, y_grid, z_grid), Ez_sphere)

def e_field_ode(t, pos):
    try:
        E_vec = np.array([interp_Ex(pos)[0], interp_Ey(pos)[0], interp_Ez(pos)[0]])
        E_mag = np.linalg.norm(E_vec)
        if E_mag < 1e-6:
            return np.zeros(3)
        return E_vec / E_mag
    except ValueError: # Out of interpolation bounds
        return np.zeros(3)

# Events to stop integration
def hit_sphere(t, pos):
    return np.linalg.norm(pos) - sphere_r
hit_sphere.terminal = True
hit_sphere.direction = -1

def hit_torus(t, pos):
    # avoid immediate stop at t=0 (start points are on torus surface)
    if t < 1e-6:
        return 1.0
    x0, y0, z0 = pos
    torus_eq = (np.sqrt(x0**2 + y0**2) - R)**2 + z0**2 - r**2
    return torus_eq
hit_torus.terminal = True
hit_torus.direction = -1

def leave_box(t, pos):
    return max(abs(p) for p in pos) - N/2
leave_box.terminal = False


# Starting points for streamlines on the torus surface
start_points = []
for i in range(0, 360, 30):
    for j in range(0, 360, 30):
        u_val, v_val = np.deg2rad(i), np.deg2rad(j)
        start_x = (R + r * np.cos(v_val)) * np.cos(u_val)
        start_y = (R + r * np.cos(v_val)) * np.sin(u_val)
        start_z = r * np.sin(v_val)
        start_points.append([start_x, start_y, start_z])

t_span = [0, 1000] # Max "time" or arc length
for start_point in start_points:
    sol = solve_ivp(
        e_field_ode, t_span, start_point,
        method='RK45', dense_output=True,
        events=(hit_sphere, hit_torus, leave_box),
        rtol=1e-6, atol=1e-6
    )
    ax.plot(sol.y[0], sol.y[1], sol.y[2], color='r', alpha=0.7)


ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Electric Field of a Torus and a Sphere (RK45)')

# Set aspect ratio to be equal and zoom out
zoom_factor = 1.5
max_range = np.array([xx.max()-xx.min(), yy.max()-yy.min(), zz.max()-zz.min()]).max() / 2.0 * zoom_factor
mid_x = (xx.max()+xx.min()) * 0.5
mid_y = (yy.max()+yy.min()) * 0.5
mid_z = (zz.max()+zz.min()) * 0.5
ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)

plt.show()

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

# --- Animation Setup ---

# Pre-calculate all streamline solutions
t_span = [0, 1000]
streamline_sols = []
for start_point in start_points:
    sol = solve_ivp(
        e_field_ode, t_span, start_point,
        method='RK45', dense_output=True,
        events=(hit_sphere, hit_torus, leave_box),
        rtol=1e-5, atol=1e-5
    )
    streamline_sols.append(sol)

# Create the figure for the animation
fig_anim = plt.figure(figsize=(10, 8))
ax_anim = fig_anim.add_subplot(111, projection='3d')

# Set plot limits and labels
ax_anim.set_xlabel('X')
ax_anim.set_ylabel('Y')
ax_anim.set_zlabel('Z')
ax_anim.set_title('Electric Field Streamline Animation')
max_range = np.array([xx.max()-xx.min(), yy.max()-yy.min(), zz.max()-zz.min()]).max() / 2.0 * 1.5
mid_x = (xx.max()+xx.min()) * 0.5
mid_y = (yy.max()+yy.min()) * 0.5
mid_z = (zz.max()+zz.min()) * 0.5
ax_anim.set_xlim(mid_x - max_range, mid_x + max_range)
ax_anim.set_ylim(mid_y - max_range, mid_y + max_range)
ax_anim.set_zlim(mid_z - max_range, mid_z + max_range)


# Plot static surfaces (torus and sphere)
ax_anim.plot_surface(x_torus, y_torus, z_torus, color='b', alpha=0.2)
ax_anim.plot_surface(x_sphere, y_sphere, z_sphere, color='g', alpha=0.4)

# Initialize line objects for the animation
lines = [ax_anim.plot([], [], [], color='r', alpha=0.7)[0] for _ in streamline_sols]

# Animation update function
def update(frame, sols, lines):
    num_points = 100 # Number of points to plot per streamline
    for sol, line in zip(sols, lines):
        # Determine the points to plot for the current frame
        t_vals = np.linspace(sol.t.min(), sol.t.max(), num_points)
        t_end_index = int(frame / 100 * num_points)
        if t_end_index > 0:
            t_current = t_vals[:t_end_index]
            points = sol.sol(t_current)
            line.set_data(points[0], points[1])
            line.set_3d_properties(points[2])
    return lines

# Create and display the animation
ani = animation.FuncAnimation(
    fig_anim, update, frames=1000,
    fargs=(streamline_sols, lines),
    blit=True, interval=10
)

plt.close(fig_anim) # Prevent static plot from showing
HTML(ani.to_jshtml())